# RL Post-Training for PlasmidLM-MoE

Compare three RL algorithms on the PlasmidLM kmer6 Mixture-of-Experts model:

| Algorithm | Advantage Estimate | Key Idea |
|---|---|---|
| **GRPO** | Group-relative (z-score within prompt group) | No value model needed — generates G completions per prompt |
| **PPO** | Reward − EMA baseline, clipped surrogate | Classic RLHF workhorse with per-token clipping |
| **DPO** | N/A (preference pairs from scorer) | Offline — no rollouts during training |

All three use the **motif alignment reward**: Smith-Waterman alignment of generated DNA against expected functional motifs (resistance genes, origins of replication, promoters, etc.).

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q "transformers>=5.2.0" parasail Biopython

import copy, re, math
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

import parasail
from Bio.Seq import Seq

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

DATA_DIR = Path("drive/MyDrive/plasmidgpt-v2-data")

print(f"Device: {DEVICE} | dtype: {DTYPE}")
!ls {DATA_DIR}

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 138.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 149.5 MB/s eta 0:00:00
Device: cuda | dtype: torch.bfloat16
motif_registry_combined_big.parquet  output			val
motif_registry_combined.parquet      outputs			vocab.json
motif_registry.json		     train
motif_registry.parquet		     training_pairs_v4.parquet


## 1. Model & Tokenizer

Load the kmer6 MoE model from HuggingFace Hub (or a local checkpoint). The reference model is a frozen deep copy used for KL regularization.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "McClain/PlasmidLM-kmer6-MoE"  # swap for local checkpoint path if needed

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = "<PAD>"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=DTYPE
).to(DEVICE).train()

ref_model = copy.deepcopy(model).eval()
for p in ref_model.parameters():
    p.requires_grad = False

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Policy model: {n_params:,} trainable params")
print(f"MoE: {getattr(model.config, 'use_moe', False)}, "
      f"experts: {getattr(model.config, 'num_experts', 'N/A')}")

`torch_dtype` is deprecated! Use `dtype` instead!


modeling_plasmid_lm.py: 0.00B [00:00, ?B/s]

moe.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/McClain/PlasmidLM-kmer6-MoE:
- moe.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/McClain/PlasmidLM-kmer6-MoE:
- modeling_plasmid_lm.py
- moe.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/313M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

Policy model: 78,324,096 trainable params
MoE: True, experts: 6


## 2. Motif Registry & Scorer

Load the motif registry — a database mapping annotation tokens (e.g. `<AMR_AMPICILLIN>`, `<ORI_COLE1>`) to their reference DNA/protein sequences. Then define the `MotifScorer` class which uses Smith-Waterman alignment (DNA + 6-frame protein) to score how well generated DNA matches expected functional motifs.

In [5]:
motifs = pd.read_parquet(DATA_DIR / "motif_registry_combined_big.parquet")
motif_records = motifs.to_dict("records")
known_tokens = set(motifs["token"].unique())
print(f"Motif registry: {len(motifs)} entries, {len(known_tokens)} unique tokens")
motifs.head()

Motif registry: 660 entries, 57 unique tokens


,uuid,token,category,features,plasmid_count,sseqid,db_source,seq_type,seq_len,sequence
0,d053544e-a84d-5811-8181-c761bd39e531,<AMR_AMPICILLIN>,AMR,"AmpR,ampR",99829,AmpR,snapgene,dna,861.0,ATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCCTTTTTTGCGG...
1,d053544e-a84d-5811-8181-c761bd39e531,<AMR_AMPICILLIN>,AMR,"AmpR,ampR",99829,AmpR_(2),snapgene,dna,861.0,ATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCCTTTTTTGCGG...
2,d053544e-a84d-5811-8181-c761bd39e531,<AMR_AMPICILLIN>,AMR,"AmpR,ampR",99829,AmpR_(3),snapgene,dna,861.0,ATGAGCATCCAACATTTTCGTGTCGCACTCATTCCCTTCTTTGCGG...
3,d053544e-a84d-5811-8181-c761bd39e531,<AMR_AMPICILLIN>,AMR,"AmpR,ampR",99829,AmpR_(4),snapgene,dna,861.0,ATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCCTTTTTTGCGG...
4,d053544e-a84d-5811-8181-c761bd39e531,<AMR_AMPICILLIN>,AMR,"AmpR,ampR",99829,P12529,swissprot,protein,291.0,MTRSYIPLNSLRAFEAAARHLSFTRAAIELNVTHSAISQHVKSLEQ...


In [7]:
class MotifScorer:
    """Score a generated DNA sequence against expected functional motifs.

    Uses semi-global Smith-Waterman alignment (parasail) with a three-pass
    pipeline: k-mer pre-filter → score-only fast check → full trace alignment.
    Supports both DNA and 6-frame protein alignments for CDS motifs.
    """

    def __init__(self, target_dna, dna_gap_open=5, dna_gap_extend=2,
                 prot_gap_open=11, prot_gap_extend=1):
        self.target_dna = self._clean_seq(target_dna)
        self.target_len = len(self.target_dna)
        self.dna_gap_open = dna_gap_open
        self.dna_gap_extend = dna_gap_extend
        self.prot_gap_open = prot_gap_open
        self.prot_gap_extend = prot_gap_extend
        self.protein_frames = self._get_six_frames(self.target_dna)
        self._target_kmers = self._kmer_set(self.target_dna, k=15) if self.target_len >= 15 else set()

    @staticmethod
    def _clean_seq(seq):
        lines = seq.strip().split('\n')
        lines = [l for l in lines if not l.startswith('>')]
        cleaned = ''.join(lines).upper()
        cleaned = re.sub(r'[^ATGCNRYSWKMBDHV]', '', cleaned)
        return cleaned

    @staticmethod
    def _kmer_set(seq, k=15):
        if len(seq) < k:
            return set()
        return {seq[i:i+k] for i in range(len(seq) - k + 1)}

    def _get_six_frames(self, dna):
        s = Seq(dna)
        frames = []
        for i in range(3):
            f_seq = s[i:]
            f_seq = f_seq[:len(f_seq) - (len(f_seq) % 3)]
            frames.append(str(f_seq.translate(stop_symbol="X")))
            r_seq = s.reverse_complement()[i:]
            r_seq = r_seq[:len(r_seq) - (len(r_seq) % 3)]
            frames.append(str(r_seq.translate(stop_symbol="X")))
        return frames

    # ── CIGAR parsing ──────────────────────────────────────────────

    @staticmethod
    def _parse_cigar(cigar_str):
        """Parse CIGAR string, stripping leading/trailing insertions."""
        ops = re.findall(r'(\d+)([MIDX=])', cigar_str)
        while ops and ops[0][1] == 'I':
            ops.pop(0)
        while ops and ops[-1][1] == 'I':
            ops.pop()

        all_ops = re.findall(r'(\d+)([MIDX=])', cigar_str)
        leading_i = 0
        for length_str, op in all_ops:
            if op == 'I':
                leading_i += int(length_str)
            else:
                break

        matches = mismatches = ins = dels = 0
        query_pos = leading_i
        ref_pos = 0
        start_query = end_query = None

        for length_str, op in ops:
            length = int(length_str)
            if op in ('=', 'M'):
                if start_query is None:
                    start_query = query_pos
                end_query = query_pos + length - 1
                matches += length
                query_pos += length
                ref_pos += length
            elif op == 'X':
                if start_query is None:
                    start_query = query_pos
                end_query = query_pos + length - 1
                mismatches += length
                query_pos += length
                ref_pos += length
            elif op == 'I':
                ins += length
                query_pos += length
            elif op == 'D':
                dels += length
                ref_pos += length

        return {
            "matches": matches, "mismatches": mismatches,
            "ins": ins, "dels": dels,
            "core_len": matches + mismatches,
            "start_query": start_query if start_query is not None else 0,
            "end_query": end_query if end_query is not None else 0,
            "ref_consumed": ref_pos,
        }

    # ── Alignment ──────────────────────────────────────────────────

    def _align_dna_score_only(self, motif_seq):
        """Fast score-only alignment — no traceback, no CIGAR."""
        res = parasail.sg_qx_striped_sat(
            self.target_dna, motif_seq,
            self.dna_gap_open, self.dna_gap_extend, parasail.dnafull
        )
        return res.score / len(motif_seq) if len(motif_seq) > 0 else 0.0

    def _align_dna(self, motif_seq):
        res = parasail.sg_qx_trace_striped_sat(
            self.target_dna, motif_seq,
            self.dna_gap_open, self.dna_gap_extend, parasail.dnafull
        )
        cigar_str = res.cigar.decode.decode()
        parsed = self._parse_cigar(cigar_str)
        return self._build_hit(res.score, parsed, len(motif_seq), seq_type="dna")

    def _align_protein(self, motif_seq):
        best_hit = None
        for idx, frame_seq in enumerate(self.protein_frames):
            if not frame_seq:
                continue
            res = parasail.sg_qx_trace_striped_sat(
                frame_seq, motif_seq,
                self.prot_gap_open, self.prot_gap_extend, parasail.blosum62
            )
            cigar_str = res.cigar.decode.decode()
            parsed = self._parse_cigar(cigar_str)
            hit = self._build_hit(res.score, parsed, len(motif_seq),
                                  seq_type="protein", frame=idx)

            is_reverse = (idx % 2 == 1)
            frame_offset = idx // 2
            dna_start = hit["start_pos"] * 3 + frame_offset
            dna_end = (hit["end_pos"] + 1) * 3 - 1 + frame_offset
            if is_reverse:
                dna_start_tmp = self.target_len - 1 - dna_end
                dna_end = self.target_len - 1 - (hit["start_pos"] * 3 + frame_offset)
                dna_start = dna_start_tmp
            hit["dna_start"] = dna_start
            hit["dna_end"] = dna_end

            if best_hit is None or hit["score"] > best_hit["score"]:
                best_hit = hit
        return best_hit

    def _build_hit(self, score, parsed, motif_len, seq_type="dna", frame=None):
        core = parsed["core_len"]
        pct_id = (parsed["matches"] / core * 100) if core > 0 else 0.0
        coverage = (parsed["matches"] / motif_len * 100) if motif_len > 0 else 0.0
        norm_score = score / motif_len if motif_len > 0 else 0.0
        hit = {
            "score": score, "norm_score": round(norm_score, 2),
            "pct_id": round(pct_id, 2), "coverage": round(coverage, 2),
            "matches": parsed["matches"], "mismatches": parsed["mismatches"],
            "internal_gaps": parsed["ins"] + parsed["dels"],
            "alignment_len": core,
            "start_pos": parsed["start_query"], "end_pos": parsed["end_query"],
            "seq_type": seq_type,
        }
        if frame is not None:
            hit["protein_frame"] = frame
        return hit

    # ── Public API ─────────────────────────────────────────────────

    @staticmethod
    def _min_pct_id_for_length(motif_len, base_min_pct_id=85.0):
        """Length-adaptive identity threshold — short motifs need stricter cutoffs."""
        penalty = 15.0 * math.exp(-motif_len / 100.0)
        return min(base_min_pct_id + penalty, 99.0)

    def run_tokens(self, tokens_to_search, motif_db, min_pct_id=85.0,
                   min_norm_score=0.0, min_coverage=50.0,
                   adaptive_id=True, kmer_prefilter=True,
                   kmer_min_overlap=0.15, norm_score_prefilter=1.0):
        """Score motifs via three-pass pipeline: k-mer filter → score-only → full trace."""
        if isinstance(motif_db, pd.DataFrame):
            motif_db = motif_db.to_dict('records')

        token_set = set(tokens_to_search)
        sub_db = [m for m in motif_db if m['token'] in token_set]
        token_hits = defaultdict(list)

        for entry in sub_db:
            motif_seq = entry.get('sequence')
            if not motif_seq:
                continue
            is_protein = (entry.get('seq_type') or 'dna').lower() == 'protein'
            if not is_protein:
                motif_seq = self._clean_seq(motif_seq)
            else:
                motif_seq = re.sub(r'\s+', '', motif_seq).upper()
            if len(motif_seq) < 10:
                continue

            # Pass 1: k-mer pre-filter (DNA only)
            if kmer_prefilter and not is_protein and len(motif_seq) >= 15:
                motif_kmers = entry.get('_kmer_cache')
                if motif_kmers is None:
                    motif_kmers = self._kmer_set(motif_seq, k=15)
                    entry['_kmer_cache'] = motif_kmers
                if len(motif_kmers) > 0:
                    overlap = len(self._target_kmers & motif_kmers) / len(motif_kmers)
                    if overlap < kmer_min_overlap:
                        continue

            # Pass 2: score-only fast check (DNA only)
            if not is_protein and norm_score_prefilter > 0:
                fast_norm = self._align_dna_score_only(motif_seq)
                if fast_norm < norm_score_prefilter:
                    continue

            # Pass 3: full trace alignment
            hit = self._align_protein(motif_seq) if is_protein else self._align_dna(motif_seq)
            if hit is None:
                continue

            effective_min_id = (self._min_pct_id_for_length(len(motif_seq), base_min_pct_id=min_pct_id)
                                if adaptive_id else min_pct_id)
            if hit["pct_id"] < effective_min_id or hit["norm_score"] < min_norm_score or hit["coverage"] < min_coverage:
                continue

            hit["token"] = entry.get('token')
            hit["db_entry"] = entry.get('sseqid', entry.get('features', ''))
            hit["motif_len"] = len(motif_seq)
            if not is_protein:
                hit["dna_start"] = hit["start_pos"]
                hit["dna_end"] = hit["end_pos"]
            token_hits[hit["token"]].append(hit)

        per_token = []
        for token in tokens_to_search:
            hits = token_hits.get(token, [])
            if hits:
                per_token.extend(self._dedup_hits(hits))
        return self._dedup_hits_global(per_token)

    @staticmethod
    def _dedup_hits(hits, overlap_frac=0.5):
        """Greedy NMS: keep best non-overlapping hits for a single token."""
        ranked = sorted(hits, key=lambda h: h["score"], reverse=True)
        kept = []
        for hit in ranked:
            h_start, h_end = hit.get("dna_start", 0), hit.get("dna_end", 0)
            h_span = max(h_end - h_start + 1, 1)
            dominated = False
            for k in kept:
                k_start, k_end = k.get("dna_start", 0), k.get("dna_end", 0)
                overlap = max(0, min(h_end, k_end) - max(h_start, k_start) + 1)
                if overlap / h_span >= overlap_frac:
                    dominated = True
                    break
            if not dominated:
                kept.append(hit)
        return kept

    @staticmethod
    def _dedup_hits_global(hits, iou_threshold=0.3):
        """Cross-token NMS: resolve overlapping hits from different tokens."""
        ranked = sorted(hits, key=lambda h: h.get("norm_score", 0), reverse=True)
        kept = []
        for hit in ranked:
            h_start, h_end = hit.get("dna_start", 0), hit.get("dna_end", 0)
            h_span = max(h_end - h_start + 1, 1)
            dominated = False
            for k in kept:
                k_start, k_end = k.get("dna_start", 0), k.get("dna_end", 0)
                k_span = max(k_end - k_start + 1, 1)
                overlap = max(0, min(h_end, k_end) - max(h_start, k_start) + 1)
                union = h_span + k_span - overlap
                if union > 0 and overlap / union >= iou_threshold:
                    dominated = True
                    break
            if not dominated:
                kept.append(hit)
        return kept

    # ── Composite scoring ──────────────────────────────────────────

    def score(self, expected_tokens, motif_db,
              w_id=0.4, w_cov=0.35, w_norm=0.25,
              norm_score_cap=5.0, sharpness=2.0,
              recall_floor=0.5, **run_kwargs):
        """Compute composite score: quality (geo-mean of found tokens) × recall penalty."""
        hits = self.run_tokens(expected_tokens, motif_db, **run_kwargs)

        best_per_token = {}
        for hit in hits:
            tok = hit["token"]
            if tok not in best_per_token or hit["norm_score"] > best_per_token[tok]["norm_score"]:
                best_per_token[tok] = hit

        found_scores = []
        for tok in expected_tokens:
            if tok in best_per_token:
                h = best_per_token[tok]
                quality = (w_id * h["pct_id"] / 100.0
                           + w_cov * h["coverage"] / 100.0
                           + w_norm * min(h["norm_score"] / norm_score_cap, 1.0))
                found_scores.append(quality)

        n_expected = len(expected_tokens)
        n_found = len(found_scores)
        recall = n_found / n_expected if n_expected > 0 else 0.0

        if found_scores:
            geo_mean = math.exp(sum(math.log(max(s, 1e-6)) for s in found_scores) / len(found_scores))
            quality = geo_mean ** sharpness
        else:
            geo_mean = quality = 0.0

        recall_penalty = recall_floor + (1.0 - recall_floor) * recall
        composite = quality * recall_penalty

        return {
            "composite": round(composite, 4),
            "quality": round(quality, 4),
            "geo_mean": round(geo_mean, 4),
            "recall": round(recall, 4),
            "found": n_found,
            "expected": n_expected,
            "hits": hits,
        }

print(f"MotifScorer defined — {len(MotifScorer.__dict__)} methods")

MotifScorer defined — 18 methods


In [8]:
HARD_PREFIXES = {"AMR_", "ORI_", "PROM_", "REPORTER_", "REP_", "TAG_", "ELEM_"}
EXCLUDE_TOKENS = {"<ELEM_IRES>", "<ELEM_TRACRRNA>"}


def parse_hard_tokens(prompt):
    """Extract scorable motif tokens from a prompt string."""
    all_tokens = re.findall(r"<[^>]+>", prompt)
    return [t for t in all_tokens
            if any(t.strip("<>").startswith(p) for p in HARD_PREFIXES)
            and t in known_tokens and t not in EXCLUDE_TOKENS]


def score_batch(prompts, completions, eos_bonus=0.15):
    """Score a batch of (prompt, completion) pairs using MotifScorer.

    Returns list of scalar rewards suitable for RL training.
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        has_eos = "<EOS>" in completion or "</s>" in completion
        dna = re.sub(r"<[^>]+>", "", completion.upper())
        dna = re.sub(r"[^ATGCN]", "", dna)
        hard_tokens = parse_hard_tokens(prompt)

        if len(dna) < 100 or not hard_tokens:
            rewards.append(0.0)
            continue

        scorer = MotifScorer(dna)
        result = scorer.score(hard_tokens, motif_records, sharpness=3.0)
        reward = result["composite"] + (eos_bonus if has_eos else 0.0)

        if len(dna) > 3500:
            excess = (len(dna) - 3500) / 3500
            reward *= max(0.5, 1.0 - 0.3 * excess)

        rewards.append(reward)
    return rewards


# Quick sanity check
test_r = score_batch(
    ["<BOS><AMR_AMPICILLIN><ORI_COLE1><SEP>"],
    ["ATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCC" * 20 + "<EOS>"],
)
print(f"Sanity check reward: {test_r[0]:.4f}")

Sanity check reward: 0.1500


## 3. Prompt Data

Load prompts with hard annotation tokens (functional motifs the model should place). Each prompt looks like `<BOS><AMR_KANAMYCIN><ORI_COLE1><SEP>` — the model generates DNA after `<SEP>`.

In [10]:
training_pairs = pd.read_parquet(DATA_DIR / "training_pairs_v4.parquet")

# Filter to prompts that have hard annotation tokens (motifs the scorer can evaluate)
if "has_hard_tokens" in training_pairs.columns:
    training_pairs = training_pairs[training_pairs["has_hard_tokens"] == True]
elif "reward_motifs" in training_pairs.columns:
    training_pairs = training_pairs[training_pairs["reward_motifs"].astype(bool)]

prompt_col = "prompt" if "prompt" in training_pairs.columns else "token_prompt"
all_prompts = [p + "<SEP>" for p in training_pairs[prompt_col].tolist()]

print(f"Loaded {len(all_prompts)} prompts with hard tokens")
print(f"Example: {all_prompts[0]}")

Loaded 108198 prompts with hard tokens
Example: <BOS><VEC_YEAST><SP_YEAST><COPY_HIGH><AMR_AMPICILLIN><AMR_TETRACYCLINE><ORI_COLE1><ORI_F1><PROM_AMPR><PROM_CMV><PROM_LAC><PROM_T3><PROM_T7><ELEM_CMV_ENHANCER><ELEM_IRES><ELEM_TRACRRNA><SEQ><SEP>


## 4. RL Utilities

In [13]:
# ── Shared hyperparams (tweak these) ──
LR          = 5e-6
MAX_STEPS   = 50
MAX_NEW_TOK = 1024
TEMPERATURE = 1.0
TOP_P       = 0.95
MAX_GRAD    = 0.5
EOS_BONUS   = 0.15
KL_COEF     = 0.05
CLIP_RANGE  = 0.2
AUX_COEF    = 0.01


def cycling_batch_iterator(items, batch_size, seed=42):
    """Yield batches from items, reshuffling each epoch. Cycles forever."""
    rng = np.random.RandomState(seed)
    indices = np.arange(len(items))
    pos = 0
    rng.shuffle(indices)
    while True:
        if pos + batch_size > len(indices):
            batch_indices = list(indices[pos:])
            rng.shuffle(indices)
            remaining = batch_size - len(batch_indices)
            batch_indices.extend(indices[:remaining])
            pos = remaining
        else:
            batch_indices = indices[pos:pos + batch_size]
            pos += batch_size
        yield [items[i] for i in batch_indices]


def per_token_logps(mdl, full_ids, prompt_len):
    """Per-token log probs for completion tokens + MoE aux loss."""
    attn = (full_ids != tokenizer.pad_token_id).long()
    with torch.amp.autocast("cuda", dtype=DTYPE):
        hidden, _, aux = mdl.model(input_ids=full_ids, attention_mask=attn)
        logits = mdl.lm_head(hidden)
    shift_logits = logits[:, prompt_len - 1:-1, :]
    shift_labels = full_ids[:, prompt_len:]
    lp = F.log_softmax(shift_logits.float(), dim=-1)
    tok_lp = lp.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
    mask = shift_labels != tokenizer.pad_token_id
    return tok_lp * mask.float(), mask, aux


def generate_rollouts(prompts):
    """Generate → decode → score → compute log probs under policy & ref."""
    model.eval()
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    plen = enc["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=MAX_NEW_TOK, temperature=TEMPERATURE,
            do_sample=True, top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    texts = tokenizer.batch_decode(out[:, plen:], skip_special_tokens=False)
    model.train()
    with torch.no_grad():
        pt_lp, mask, _ = per_token_logps(model, out, plen)
        ref_lp, _, _   = per_token_logps(ref_model, out, plen)
    counts = mask.sum(-1).clamp(min=1)
    rewards = torch.tensor(score_batch(prompts, texts, eos_bonus=EOS_BONUS))
    return dict(
        prompts=prompts, texts=texts, full_ids=out.cpu(), prompt_len=plen,
        log_probs=(pt_lp.sum(-1) / counts).cpu(),
        ref_log_probs=(ref_lp.sum(-1) / counts).cpu(),
        pt_lp=pt_lp.cpu(), ref_lp=ref_lp.cpu(), mask=mask.cpu(),
        rewards=rewards,
    )


def train_step(rollout, advantages, algo, opt):
    """One gradient step: recompute log probs → loss → backward → clip → step."""
    ids = rollout["full_ids"].to(DEVICE)
    old = rollout["pt_lp"].to(DEVICE)
    ref = rollout["ref_lp"].to(DEVICE)
    m   = rollout["mask"].to(DEVICE)
    adv = advantages.to(DEVICE)

    model.train()
    opt.zero_grad()
    cur, _, aux = per_token_logps(model, ids, rollout["prompt_len"])
    loss = algo.compute_loss(cur, old, adv, ref, m)
    if aux is not None:
        loss = loss + AUX_COEF * aux
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD)
    opt.step()

    with torch.no_grad():
        kl = (torch.exp(ref - cur) - (ref - cur) - 1.0)
        kl = ((kl * m.float()).sum(-1) / m.sum(-1).clamp(min=1)).mean()
    return {"loss": loss.item(), "kl": kl.item()}


def reset_policy():
    """Reset policy to reference weights (clean slate between algorithms)."""
    model.load_state_dict(ref_model.state_dict())
    model.train()
    print("Policy reset to reference weights")

---
## 5. GRPO
G completions per prompt → z-score rewards within each group → clipped PG + KL penalty.

In [ ]:
class GRPO:
    """Group Relative Policy Optimization (DeepSeek-style).

    Generates G completions per prompt, z-scores rewards within each group,
    then applies a clipped policy gradient with KL penalty.
    """

    def __init__(self, kl_coef=0.05, cliprange=0.2, num_generations=4):
        self.kl_coef = kl_coef
        self.cliprange = cliprange
        self.num_generations = num_generations

    def compute_advantages(self, rewards, log_probs, ref_log_probs):
        G = self.num_generations
        grouped = rewards.view(-1, G)
        mean = grouped.mean(dim=1, keepdim=True)
        std = grouped.std(dim=1, keepdim=True)
        valid = std > 0.1
        advantages = torch.where(valid, (grouped - mean) / std.clamp(min=0.1),
                                 torch.zeros_like(grouped))
        return advantages.view(-1)

    def compute_loss(self, per_token_logps, old_per_token_logps, advantages,
                     ref_per_token_logps, mask):
        ratio = torch.exp(per_token_logps - old_per_token_logps)
        clipped = torch.clamp(ratio, 1 - self.cliprange, 1 + self.cliprange)

        adv = advantages.detach().unsqueeze(1)
        per_token_loss = -torch.min(adv * ratio, adv * clipped)

        log_ratio = ref_per_token_logps - per_token_logps
        per_token_kl = torch.exp(log_ratio.clamp(max=5.0)) - log_ratio - 1.0
        per_token_loss = per_token_loss + self.kl_coef * per_token_kl

        return ((per_token_loss * mask.float()).sum(-1) / mask.sum(-1).clamp(min=1)).mean()


G = 4  # completions per prompt

grpo = GRPO(kl_coef=KL_COEF, cliprange=CLIP_RANGE, num_generations=G)

reset_policy()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
prompt_iter = cycling_batch_iterator(all_prompts, batch_size=4, seed=42)

grpo_log = []
for step in tqdm(range(MAX_STEPS), desc="GRPO"):
    prompts = [p for p in next(prompt_iter) for _ in range(G)]  # 4 prompts × 4 gens = 16
    rollout = generate_rollouts(prompts)
    adv = grpo.compute_advantages(rollout["rewards"], rollout["log_probs"], rollout["ref_log_probs"])
    metrics = train_step(rollout, adv, grpo, opt)
    metrics["reward"] = rollout["rewards"].mean().item()
    grpo_log.append(metrics)
    if (step + 1) % 10 == 0:
        r = np.mean([m["reward"] for m in grpo_log[-10:]])
        print(f"  step {step+1:3d} | reward {r:.3f} | KL {metrics['kl']:.4f} | loss {metrics['loss']:.4f}")

Policy reset to reference weights


GRPO:   0%|          | 0/50 [00:00<?, ?it/s]

---
## 6. PPO
Same clipped surrogate, but advantages are `reward − EMA_baseline` instead of group-relative. No value network.

In [ ]:
class PPO:
    """PPO-clip with EMA reward baseline (no value network)."""

    def __init__(self, kl_coef=0.05, cliprange=0.2, ema=0.95):
        self.kl_coef, self.cliprange, self.ema = kl_coef, cliprange, ema
        self._bl = None

    def compute_advantages(self, rewards, log_probs, ref_log_probs):
        mu = rewards.mean().item()
        self._bl = mu if self._bl is None else self.ema * self._bl + (1 - self.ema) * mu
        adv = rewards - self._bl
        return adv / adv.std().clamp(min=0.1)

    def compute_loss(self, cur, old, adv, ref, mask):
        ratio = torch.exp(cur - old)
        clipped = ratio.clamp(1 - self.cliprange, 1 + self.cliprange)
        a = adv.detach().unsqueeze(1)
        pg = -torch.min(a * ratio, a * clipped)
        kl = torch.exp(ref - cur) - (ref - cur) - 1.0
        loss = pg + self.kl_coef * kl
        return ((loss * mask.float()).sum(-1) / mask.sum(-1).clamp(min=1)).mean()

In [ ]:
ppo = PPO(kl_coef=KL_COEF, cliprange=CLIP_RANGE)

reset_policy()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
prompt_iter = cycling_batch_iterator(all_prompts, batch_size=8, seed=123)

ppo_log = []
for step in tqdm(range(MAX_STEPS), desc="PPO"):
    rollout = generate_rollouts(next(prompt_iter))
    adv = ppo.compute_advantages(rollout["rewards"], rollout["log_probs"], rollout["ref_log_probs"])
    metrics = train_step(rollout, adv, ppo, opt)
    metrics["reward"] = rollout["rewards"].mean().item()
    ppo_log.append(metrics)
    if (step + 1) % 10 == 0:
        r = np.mean([m["reward"] for m in ppo_log[-10:]])
        print(f"  step {step+1:3d} | reward {r:.3f} | KL {metrics['kl']:.4f} | loss {metrics['loss']:.4f}")

---
## 7. DPO
Generate N completions per prompt → score → pick best/worst → train with preference loss. No online rollouts during training.

In [ ]:
DPO_BETA = 0.1
N_CANDIDATES = 4  # generate N per prompt, pick best/worst


def make_preference_pairs(prompts):
    """Generate N completions per prompt → score → return (chosen, rejected) pairs."""
    expanded = [p for p in prompts for _ in range(N_CANDIDATES)]
    model.eval()
    enc = tokenizer(expanded, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    plen = enc["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=MAX_NEW_TOK, temperature=TEMPERATURE,
            do_sample=True, top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    texts = tokenizer.batch_decode(out[:, plen:], skip_special_tokens=False)
    rewards = score_batch(expanded, texts, eos_bonus=EOS_BONUS)

    pairs = []
    for i, prompt in enumerate(prompts):
        grp = [(j, rewards[j]) for j in range(i * N_CANDIDATES, (i + 1) * N_CANDIDATES)]
        best = max(grp, key=lambda x: x[1])
        worst = min(grp, key=lambda x: x[1])
        if best[1] > worst[1]:
            pairs.append(dict(
                chosen=out[best[0]].cpu(), rejected=out[worst[0]].cpu(),
                prompt_len=plen, chosen_r=best[1], rejected_r=worst[1],
            ))
    return pairs


def dpo_loss_pair(chosen_ids, rejected_ids, plen):
    """DPO loss for one (chosen, rejected) pair."""
    def slp(mdl, ids):
        lp, mask, aux = per_token_logps(mdl, ids.unsqueeze(0).to(DEVICE), plen)
        return (lp * mask.float()).sum() / mask.sum().clamp(min=1), aux

    pi_w, aux_w = slp(model, chosen_ids)
    pi_l, aux_l = slp(model, rejected_ids)
    with torch.no_grad():
        ref_w, _ = slp(ref_model, chosen_ids)
        ref_l, _ = slp(ref_model, rejected_ids)

    loss = -F.logsigmoid(DPO_BETA * ((pi_w - ref_w) - (pi_l - ref_l)))
    aux = (aux_w + aux_l) / 2 if aux_w is not None else None
    return loss, aux

In [ ]:
reset_policy()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
prompt_iter = cycling_batch_iterator(all_prompts, batch_size=8, seed=456)

dpo_log = []
for step in tqdm(range(MAX_STEPS), desc="DPO"):
    pairs = make_preference_pairs(next(prompt_iter))
    if not pairs:
        continue

    model.train()
    opt.zero_grad()
    total_loss = 0.0
    for p in pairs:
        loss, aux = dpo_loss_pair(p["chosen"], p["rejected"], p["prompt_len"])
        scaled = loss / len(pairs)
        if aux is not None:
            scaled = scaled + AUX_COEF * aux / len(pairs)
        scaled.backward()
        total_loss += loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD)
    opt.step()

    chosen_r = np.mean([p["chosen_r"] for p in pairs])
    margin = np.mean([p["chosen_r"] - p["rejected_r"] for p in pairs])
    dpo_log.append({"loss": total_loss / len(pairs), "chosen_reward": chosen_r, "margin": margin})

    if (step + 1) % 10 == 0:
        recent = dpo_log[-10:]
        print(f"  step {step+1:3d} | loss {np.mean([m['loss'] for m in recent]):.4f} | "
              f"margin {np.mean([m['margin'] for m in recent]):.3f}")

---
## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot([m["reward"] for m in grpo_log], label="GRPO", alpha=0.7)
axes[0].plot([m["reward"] for m in ppo_log], label="PPO", alpha=0.7)
axes[0].plot([m["chosen_reward"] for m in dpo_log], label="DPO (chosen)", alpha=0.7, ls="--")
axes[0].set(xlabel="Step", ylabel="Reward", title="Reward")
axes[0].legend()

axes[1].plot([m["loss"] for m in grpo_log], label="GRPO", alpha=0.7)
axes[1].plot([m["loss"] for m in ppo_log], label="PPO", alpha=0.7)
axes[1].plot([m["loss"] for m in dpo_log], label="DPO", alpha=0.7, ls="--")
axes[1].set(xlabel="Step", ylabel="Loss", title="Loss")
axes[1].legend()

axes[2].plot([m["kl"] for m in grpo_log], label="GRPO", alpha=0.7)
axes[2].plot([m["kl"] for m in ppo_log], label="PPO", alpha=0.7)
axes[2].set(xlabel="Step", ylabel="KL(π || π_ref)", title="KL Divergence")
axes[2].legend()

fig.suptitle("RL Algorithm Comparison — PlasmidLM-MoE", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. Sanity Check

In [ ]:
test_prompts = [
    "<BOS><AMR_KANAMYCIN><ORI_COLE1><SEP>",
    "<BOS><AMR_AMPICILLIN><ORI_COLE1><PROM_LAC><SEP>",
]

model.eval()
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=1024,
            temperature=0.8, do_sample=True, top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )
    completion = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    dna_only = re.sub(r"<[^>]+>", "", completion)

    reward = score_batch([prompt], [completion])[0]
    print(f"Prompt: {prompt}")
    print(f"  Length: {len(dna_only)} bp | Reward: {reward:.3f}")
    print(f"  First 200 bp: {dna_only[:200]}...")
    print()